# 第 6 章 · 相关与线性回归

**相关**衡量两个变量一起变动的方向和强度；**回归**用一条线去预测。
注意：相关不等于因果。

## 0. 准备环境

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 让 matplotlib 正确显示中文与负号
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei"]
plt.rcParams["axes.unicode_minus"] = False

np.random.seed(42)   # 固定随机种子，保证每次运行结果一致
print("环境就绪：numpy", np.__version__, "| pandas", pd.__version__)
import scipy.stats as st

## 1. 协方差与相关系数

In [ ]:
df = pd.read_csv("data/height_weight.csv")
x = df["身高_cm"].values
y = df["体重_kg"].values

cov = np.cov(x, y, ddof=1)[0, 1]                 # 样本协方差
pearson_r, p_pearson = st.pearsonr(x, y)         # 皮尔逊相关
spearman_r, p_spearman = st.spearmanr(x, y)      # 斯皮尔曼秩相关

print("协方差       = %.3f  (受量纲影响，不好直接解释)" % cov)
print("皮尔逊 r     = %.3f  (p = %.3g)" % (pearson_r, p_pearson))
print("斯皮尔曼 ρ   = %.3f  (p = %.3g)" % (spearman_r, p_spearman))
print("r² = %.3f -> 身高可解释体重变异的 %.1f%%" % (pearson_r ** 2, pearson_r ** 2 * 100))

**皮尔逊 r** 的取值范围 [−1, 1]：
- r > 0 正相关，r < 0 负相关，r ≈ 0 无线性相关
- |r| 越接近 1 越强
- r 只度量**线性**关系；非线性关系可能 r≈0 却强相关（见下）

In [ ]:
# 非线性关系：r≈0 但存在强关系
t = np.linspace(-3, 3, 200)
u = t
v = t ** 2 + np.random.normal(0, 0.3, 200)
print("U 与 V 的皮尔逊 r = %.3f（但其实是抛物线关系）" % st.pearsonr(u, v)[0])

plt.scatter(u, v, s=12, color="#2255aa")
plt.title("非线性关系：r 接近 0，但关系很强")
plt.xlabel("U"); plt.ylabel("V"); plt.show()

## 2. 散点图：先看图，再算数

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(x, y, s=20, alpha=0.7, color="#6da8f5")
plt.title("身高 vs 体重")
plt.xlabel("身高 (cm)"); plt.ylabel("体重 (kg)")
plt.show()

## 3. 简单线性回归

模型：y = b₀ + b₁·x + ε

最小二乘法最小化残差平方和，得到斜率与截距：
b₁ = r·(s_y / s_x)， b₀ = ȳ − b₁·x̄

In [ ]:
b1 = pearson_r * (y.std(ddof=1) / x.std(ddof=1))   # 斜率
b0 = y.mean() - b1 * x.mean()                       # 截距
print("回归方程: 体重 = %.3f + %.3f × 身高" % (b0, b1))
print("解释：身高每增加 1 cm，体重平均增加 %.3f kg" % b1)

# 与 numpy 的多项式拟合对照
slope, intercept = np.polyfit(x, y, 1)
print("numpy 拟合结果 slope=%.3f, intercept=%.3f（一致）" % (slope, intercept))

## 4. 拟合优度 R² 与残差

In [ ]:
y_pred = b0 + b1 * x
residuals = y - y_pred

ss_res = np.sum(residuals ** 2)              # 残差平方和
ss_tot = np.sum((y - y.mean()) ** 2)         # 总平方和
r2 = 1 - ss_res / ss_tot

print("R² = %.4f，模型解释了 %.1f%% 的体重变异" % (r2, r2 * 100))
print("残差标准差 (RMSE) = %.2f kg" % math.sqrt(ss_res / (len(y) - 2)))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(x, y, s=18, alpha=0.7, label="实际值")
axes[0].plot(x, y_pred, color="red", label="回归线")
axes[0].set_title("线性回归拟合"); axes[0].set_xlabel("身高"); axes[0].set_ylabel("体重")
axes[0].legend()

axes[1].scatter(y_pred, residuals, s=18, alpha=0.7, color="#f5a06d")
axes[1].axhline(0, color="black", linestyle="--")
axes[1].set_title("残差图（应随机散布在 0 附近）")
axes[1].set_xlabel("预测值"); axes[1].set_ylabel("残差")

plt.tight_layout(); plt.show()

## 5. 用真实成绩数据做回归

研究数学成绩能否预测英语成绩。

In [ ]:
df2 = pd.read_csv("data/students.csv")
r, p = st.pearsonr(df2["数学"], df2["英语"])
s, i = np.polyfit(df2["数学"], df2["英语"], 1)
print("数学与英语相关 r = %.3f (p=%.3g)" % (r, p))
print("回归方程: 英语 = %.2f + %.2f × 数学" % (i, s))

## 小结

- 相关：度量方向与强度；回归：建立预测方程。
- 皮尔逊 r 只看线性关系，别忘先画散点图。
- R² 表示模型解释的变异比例；残差图用于检查模型是否合适。
- **相关 ≠ 因果**：可能由第三个变量（混杂因素）造成。